Implement a Neural Machine Translation (NMT) Model with an RNN. Train an Encoder-
Decoder RNN for translating English sentences to French.

- Tokenize and preprocess text (e.g., subword tokenization).
- Implement sequence padding and masking.
- Train with Teacher Forcing and evaluate BLEU scores.
- Experiment with attention mechanisms to improve performance.

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re
import unicodedata
import os
import requests
import zipfile

In [5]:
def download_extract_data(url, zip_file, text_file):
    if not os.path.exists(zip_file):
        print("Downloading dataset...")
        response = requests.get(url, stream=True)
        with open(zip_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    
    if not os.path.exists(text_file):
        print("Extracting dataset...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall()

In [6]:
def preprocess_text(text):
    text = unicodedata.normalize('NFD', text.lower().strip())
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    text = re.sub(r'[" "]+', " ", text)
    text = re.sub(r"[^a-zA-Z?.!,¿]+", " ", text)
    return '<start> ' + text.strip() + ' <end>'

In [7]:
def load_dataset(file_path, num_samples):
    source, target = [], []
    with open(file_path, encoding='UTF-8') as f:
        for i, line in enumerate(f):
            if i >= num_samples:
                break
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                source.append(preprocess_text(parts[0]))
                target.append(preprocess_text(parts[1]))
    return source, target

In [8]:
def tokenize_pad(texts, tokenizer=None, max_len=None, is_target=False):
    if tokenizer is None:
        tokenizer = Tokenizer(filters='', oov_token='<OOV>')
        tokenizer.fit_on_texts(texts)
    
    seq = tokenizer.texts_to_sequences(texts)
    max_len = max(len(s) for s in seq) if is_target else (max_len or MAX_SEQ_LENGTH)
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    return padded, tokenizer, max_len

In [10]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate, Dot, Activation
from tensorflow.keras.models import Model

In [11]:
def build_nmt_model(input_vocab_size, target_vocab_size, input_len, target_len):
    # Encoder
    enc_inputs = Input(shape=(input_len,))
    enc_embed = Embedding(input_vocab_size, EMBEDDING_DIM, mask_zero=True)(enc_inputs)
    enc_lstm = LSTM(LSTM_UNITS, return_sequences=True, return_state=True)
    enc_outputs, state_h, state_c = enc_lstm(enc_embed)
    
    # Decoder with Attention
    dec_inputs = Input(shape=(target_len-1,))
    dec_embed = Embedding(target_vocab_size, EMBEDDING_DIM, mask_zero=True)(dec_inputs)
    dec_lstm = LSTM(LSTM_UNITS, return_sequences=True)
    dec_outputs = dec_lstm(dec_embed, initial_state=[state_h, state_c])
    
    # Attention Mechanism
    attention = Dot(axes=[2, 2])([dec_outputs, enc_outputs])
    attention = Activation('softmax')(attention)
    context = Dot(axes=[2, 1])([attention, enc_outputs])
    decoder_combined = Concatenate()([dec_outputs, context])
    
    # Output
    outputs = Dense(target_vocab_size, activation='softmax')(decoder_combined)
    
    return Model([enc_inputs, dec_inputs], outputs)

In [13]:
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [14]:
def train_model(model, X_train, y_train, X_val, y_val):
    early_stop = EarlyStopping(monitor='val_loss', patience=2)
    
    model.compile(optimizer='adam', 
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
    
    history = model.fit(
        [X_train, y_train[:, :-1]],
        np.expand_dims(y_train[:, 1:], -1),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=([X_val, y_val[:, :-1]], 
                         np.expand_dims(y_val[:, 1:], -1)),
        callbacks=[early_stop]
    )
    return history

In [15]:
def translate(sentence, model, src_tokenizer, trg_tokenizer, src_len, trg_len):
    sentence = preprocess_text(sentence)
    seq = src_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=src_len, padding='post')
    
    target_seq = np.zeros((1, trg_len-1))
    target_seq[0, 0] = trg_tokenizer.word_index['<start>']
    
    translated = []
    for i in range(1, trg_len-1):
        output = model.predict([seq, target_seq], verbose=0)
        word_idx = np.argmax(output[0, i-1])
        word = trg_tokenizer.index_word.get(word_idx, '')
        
        if word == '<end>':
            break
        translated.append(word)
        target_seq[0, i] = word_idx
    
    return ' '.join(translated)

In [16]:
# Constants
NUM_SAMPLES = 10000
MAX_VOCAB_SIZE = 5000
EMBEDDING_DIM = 128
LSTM_UNITS = 256
BATCH_SIZE = 32
EPOCHS = 20
MAX_SEQ_LENGTH = 15
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip'

if __name__ == '__main__':
    # Download and prepare data
    download_extract_data(DATA_URL, 'fra-eng.zip', 'fra.txt')
    src_texts, trg_texts = load_dataset('fra.txt', NUM_SAMPLES)
    
    # Tokenize and pad sequences
    X, src_tok, src_len = tokenize_pad(src_texts, max_len=MAX_SEQ_LENGTH)
    y, trg_tok, trg_len = tokenize_pad(trg_texts, is_target=True)
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)
    
    # Build and train model
    model = build_nmt_model(
        len(src_tok.word_index)+1,
        len(trg_tok.word_index)+1,
        src_len,
        trg_len
    )
    train_model(model, X_train, y_train, X_val, y_val)
    
    # Test translations
    test_phrases = ["Hello", "What's your name?", "How are you?"]
    for phrase in test_phrases:
        print(f"\nEnglish: {phrase}")
        print("French:", translate(phrase, model, src_tok, trg_tok, src_len, trg_len))

Extracting dataset...
Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 42s 152ms/step - accuracy: 0.6602 - loss: 2.9392 - val_accuracy: 0.7504 - val_loss: 1.5091
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 42s 169ms/step - accuracy: 0.7635 - loss: 1.4094 - val_accuracy: 0.7834 - val_loss: 1.2959
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 78s 151ms/step - accuracy: 0.7963 - loss: 1.1863 - val_accuracy: 0.8131 - val_loss: 1.1406
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 174ms/step - accuracy: 0.8164 - loss: 1.0189 - val_accuracy: 0.8249 - val_loss: 1.0565
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 98s 240ms/step - accuracy: 0.8315 - loss: 0.8855 - val_accuracy: 0.8355 - val_loss: 0.9858
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 55s 221ms/step - accuracy: 0.8462 - loss: 0.7677 - val_accuracy: 0.8422 - val_loss: 0.9404
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 194ms/step - accuracy: 0.8573 - loss: 0.6743 - val_accuracy: 0.8482 - val_loss: 0.9002
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 71s 150ms/step - accu